In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

tools = await client.get_tools()

In [3]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow up questions."
)

In [4]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "abcd1234"}}

response =  await agent.ainvoke(
    {"messages": [HumanMessage(content="Get me a direct flight from San Francisco to Tokyo on June 30th")]},
    config
    )

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Get me a direct flight from San Francisco to Tokyo on June 30th', additional_kwargs={}, response_metadata={}, id='d0682a6a-6121-49ab-a336-4eb8f171a363'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 695, 'prompt_tokens': 1229, 'total_tokens': 1924, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 640, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1024}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DoojqLQwBxCsJoCaDCjJCpeJv9hkb', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019eac19-7eee-7562-b7d8-e009e25dec16-0', tool_calls=[{'name': 'search-flight', 'args': {'flyFrom': 'San Francisco', 'flyTo': 'Tokyo', 'departureDate': '30/06/2026', 'departureDateFlexRange

In [6]:
print(response["messages"][-1].content)

Here are the direct flight options from San Francisco (SFO) to Tokyo on June 30 (June 30, 2026), grouped as requested.

Cheapest direct option
| Route | Times (local) | Cabin | Price | Book |
|---|---|---|---:|---|
| SFO → NRT | 30/06 16:45 → 01/07 20:00 (11h 15m) | Economy | USD 733 | [Book](https://on.kiwi.com/XEgeDk) |

Shortest direct option
| Route | Times (local) | Cabin | Price | Book |
|---|---|---|---:|---|
| SFO → NRT | 30/06 12:25 → 01/07 15:00 (10h 35m) | Economy | USD 1,157 | [Book](https://on.kiwi.com/iilpfv) |

Note: These are the only direct SFO–NRT options Kiwi shows for June 30. If you’d like, I can also look for near-direct options (1-stop) or expand the date window a bit.

Have a great trip! Fun fact: Tokyo’s Shibuya Crossing is famous for its sea of pedestrians and neon signs—it's said to be one of the busiest crossings in the world.


In [15]:

response1 =  await agent.ainvoke(
    {"messages": [HumanMessage(content="okay show me the 1 stop flight options.")]},
    config
    )

In [16]:
response1["messages"][-1].content

'Here are the 1-stop flight options from San Francisco (SFO) to Tokyo on June 30, 2026 (via Taipei, TPE).\n\n1-stop options (price group: USD 989)\n| Route | Times (local) | Cabin | Price | Book |\n|---|---|---|---:|---|\n| SFO → NRT via TPE | 30/06 01:05 → 01/07 13:15 (20h 10m) | Economy | USD 989 | [Book](https://on.kiwi.com/rfWbzr) |\n| SFO → NRT via TPE | 30/06 01:05 → 01/07 19:00 (25h 55m) | Economy | USD 989 | [Book](https://on.kiwi.com/EQTzlF) |\n\nNotes:\n- Both options are 1-stop itineraries via Taipei (TPE) to Tokyo Narita (NRT) on June 30, 2026.\n- Shorter option duration is 20h 10m; longer option is 25h 55m.\n- If you’d like, I can broaden searching to include 2-stop options or other nearby dates.'